In [11]:
import os
import glob
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
MODEL = "openai/gpt-oss-120b:free"

In [4]:
knowledge_base_path = "../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)

entire_knowledge_base = ""
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read() + "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base)}")

Total characters in knowledge base: 304434


In [5]:
# Tokens.
encodings = tiktoken.encoding_for_model("gpt-oss-120b")
tokens = encodings.encode(entire_knowledge_base)
print(f"Total tokens in knowledge base: {len(tokens)}")

Total tokens in knowledge base: 63555


In [10]:
# Load the knowledge base in Langchain loaders.
folders = glob.glob("../knowledge-base/*")
documents = []

for folder in folders:
    # Get the folder name of the document (eg. "company", "contracts" etc.)
    doc_type = os.path.basename(os.path.dirname(folder))
    # Use the Langchain DirectoryLoader to load all the markdown files in the folder.
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    # Return a list of Langchain Document objects, where each document is a markdown file. Add the doc_type as metadata to each document.
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [13]:
# Divide into chunks using the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"First chunk: {chunks[0]}...")
print(f"Second chunk: {chunks[1]}...")

Total chunks created: 413
First chunk: page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': '..\\knowledge-base\\company\\about.md', 'doc_type': 'knowledge-base'}...
Second chunk: page_content='However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates wi